In [ ]:
# IMPORTS AND PARAMERS

import asyncio
from idun_guardian_sdk import GuardianClient
import pandas as pd
from collections import deque
import os

API_TOKEN = "idun_MWSQ4pkewAGNz8wwYzw_NsweXihLC8tIcFzah8vqqys4Nc-ALzjfTwl2"          # <-- put your valid token here
DEVICE_ADDRESS = "E0:53:73:AB:F9:05"
DATA_FOLDER = "data"
CSV_FILE = os.path.join(DATA_FOLDER, "eeg_data.csv")

RECORDING_TIMER = 20 
LED_SLEEP = False
# List for long-term storage
eeg_rows = []

# initialize
client = GuardianClient(api_token=API_TOKEN, address=DEVICE_ADDRESS)

In [ ]:
# Handle function

df_eeg = pd.DataFrame(columns=["timestamp", "ch1", "recordingId"])

def handle_data(event):
    print("Received data:", event.message)  # Debug printPRINT
    """
    Receives Guardian live EEG data and:
    - appends to a buffer for live plotting
    - appends to a list for later CSV export
    """
    msg = event.message
    recording_id = msg.get("recordingId", None)

    raw_eeg = msg.get("raw_eeg", [])
    if not raw_eeg:
        return

    for r in raw_eeg:
        timestamp = r.get("timestamp")
        ch1 = r.get("ch1")
        if timestamp is None or ch1 is None:
            continue

        # --- store for CSV ---
        eeg_rows.append({
            "timestamp": timestamp,
            "ch1": ch1,
            "recordingId": recording_id
        })

def save_csv():
    if not eeg_rows:
        print("No EEG data to save")
        return

    df = pd.DataFrame(eeg_rows)
    # Append to CSV if it exists
    df.to_csv(CSV_FILE, mode="a", index=False, header=not os.path.exists(CSV_FILE))
    print
    print(f"Saved {len(eeg_rows)} rows to {CSV_FILE}")





In [5]:
client.subscribe_live_insights(
    raw_eeg=True,
    filtered_eeg=False,
    imu=False,
    handler=handle_data
)
await client.start_recording(recording_timer=RECORDING_TIMER, led_sleep=LED_SLEEP)
save_csv()

[INFO] 2026-02-10 14:21:14,651: [CLIENT]: Starting recording
[INFO] 2026-02-10 14:21:14,652: [CLIENT]: Recording timer: 20 seconds
[INFO] 2026-02-10 14:21:14,653: [CLIENT]: Ensuring hardware version is synced with Cloud
[INFO] 2026-02-10 14:21:14,801: [CLIENT]: Hardware version already exists in the Cloud


Received data: {'idunId': 'cbac285f-6c2f-4ca4-863b-b7d1567b9deb', 'deviceId': 'E0-53-73-AB-F9-05', 'action': 'liveStreamInsights', 'sequence': 274, 'imu': None, 'recordingId': '1770729673426', 'partitioningKey': 'cbac285f-6c2f-4ca4-863b-b7d1567b9deb', 'raw_eeg': [{'timestamp': 1770729679.8799992, 'ch1': -9832.71119984522}, {'timestamp': 1770729679.883999, 'ch1': -9842.300098216547}, {'timestamp': 1770729679.887999, 'ch1': -9854.57120592251}, {'timestamp': 1770729679.8919992, 'ch1': -9866.66349967283}, {'timestamp': 1770729679.8959992, 'ch1': -9875.291273032579}, {'timestamp': 1770729679.8999991, 'ch1': -9880.85685740195}, {'timestamp': 1770729679.903999, 'ch1': -9891.228066829213}, {'timestamp': 1770729679.9079993, 'ch1': -9899.72172972223}, {'timestamp': 1770729679.9119992, 'ch1': -9892.233895329702}, {'timestamp': 1770729679.9159992, 'ch1': -9878.934607378795}, {'timestamp': 1770729679.9199991, 'ch1': -9872.609063697942}, {'timestamp': 1770729679.923999, 'ch1': -9855.174703022803}, {

[INFO] 2026-02-10 14:21:43,240: [CLIENT]: Recording Finished
[INFO] 2026-02-10 14:21:43,241: [CLIENT]: Recording ID 1770729673426


Saved 10040 rows to data\eeg_data.csv
